# Colombia en datos: recorrido de análisis

Notebook educativo para explorar los extractos agregados del DANE y la Registraduría incluidos en `data/processed/`. Cada porcentaje conserva su denominador. No se estima causalidad.

**Objetivos de aprendizaje:** cargar CSV con pandas, revisar calidad, calcular cambios en puntos porcentuales y crear una visualización interactiva.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
DATA = ROOT / 'data' / 'processed'
media = pd.read_csv(DATA / 'political_media.csv')
economia = pd.read_csv(DATA / 'economic_indicators.csv')
abstencion = pd.read_csv(DATA / 'abstention_reasons_by_region.csv')

print(f'Canales: {media.shape[0]} filas; economía: {economia.shape[0]} filas')
media.head()

## 1. Validar cobertura y denominadores

Antes de analizar, verifica qué años existen y cómo fue formulada la pregunta. En este caso, las opciones de medios se responden solo si la persona dice que se informa de política; además, puede marcar más de un canal.

In [ ]:
media.groupby('channel')['year'].agg(['min', 'max', 'nunique']).sort_values('max', ascending=False)

## 2. Calcular cambios en puntos porcentuales

Restar dos porcentajes produce puntos porcentuales. No es lo mismo que un cambio porcentual relativo.

In [ ]:
social = media[media['channel'].str.startswith('Redes sociales')].set_index('year')['share_percent']
delta_pp = social.loc[2023] - social.loc[2021]
print(f'Uso reportado de redes para informarse de política: {delta_pp:.1f} puntos porcentuales entre 2021 y 2023.')
social

## 3. Visualizar tendencias de canales

La suma de las líneas puede exceder 100% porque la pregunta permite respuestas múltiples.

In [ ]:
fig = px.line(media, x='year', y='share_percent', color='channel', markers=True,
              title='Canales usados para informarse de política (ECP)',
              labels={'year':'Año','share_percent':'Personas que seleccionaron el canal (%)','channel':'Canal'})
fig.update_layout(legend_title_text='Canal', yaxis_range=[0,100])
fig.show()

## 4. Explorar contexto económico

Cada indicador tiene periodos homólogos, pero sus frecuencias y significados son diferentes. Filtrar por indicador ayuda a no confundir las escalas.

In [ ]:
economia.pivot(index='period', columns='indicator', values='value')

In [ ]:
indicador = 'Annual CPI inflation'
serie = economia[economia['indicator'] == indicador].copy()
fig = px.line(serie, x='period', y='value', markers=True,
              title='Inflación anual: septiembre de cada año',
              labels={'period':'Periodo','value':'Variación anual (%)'})
fig.show()

## 5. Segmentar razones de abstención

Las razones son respuestas múltiples entre quienes no votaron. Compara regiones sin perder las advertencias de precisión del DANE.

In [ ]:
abstencion.pivot(index='reason', columns='region', values='share_percent').round(1)

## Cierre: qué aprendimos

- Los canales de información son indicadores descriptivos; no miden persuasión ni votos causados por una plataforma.
- El porcentaje oficial de participación y el autorreporte de encuesta usan universos distintos.
- Los indicadores económicos deben leerse con su fecha, frecuencia y unidad.
- Para avanzar hacia asociación individual, sigue `docs/microdata_extension.md`; descarga manualmente los microdatos y no los subas al repositorio.
